In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import confusion_matrix

In [ ]:
os.chdir('..')
assert os.path.basename(os.getcwd()) == "munich-accident-analysis", \
    "Need to be in the repo directory, restart the notebook and run this cell again"

In [ ]:
from explorative.conventions import custom_palette

# Test Set Prediction Analysis

In [ ]:
# Old Project Data (LLM Few Shot Labels Added)
test_set_path = "data/large_test_set.csv"
unfall_table_path = "data/D_Unfall_RData.csv"
few_shot_llm_labels_path = "data/few_shot_llm_predicted_labels_unfalltext_0_103111.csv"
finetuned_xlmr_trained_w_low_quality_predictions_path = "data/finetuned_xlmr_with_low_quality_labels_predictions.csv"
finetuned_xlmr_trained_w_high_quality_predictions_path = "data/finetuned_xlmr_with_high_quality_labels_predictions.csv"
multimodal_model_trained_w_high_quality_predictions_path = "data/multimodal_model_trained_w_high_quality_test_predictions.csv"
multimodal_model_trained_w_low_quality_predictions_path = "data/multimodal_model_trained_w_low_quality_test_predictions.csv"
tabular_model_trained_w_high_quality_predictions_path = "data/tabular_model_trained_w_high_quality_test_predictions.csv"
tabular_model_trained_w_low_quality_predictions_path = "data/tabular_model_trained_w_low_quality_test_predictions.csv"

# Translations
translated_unfalltext_table_path = "data/translated_unfalltext_0_103111.csv"

In [ ]:
test_set = pd.read_csv(test_set_path)
unfall_table = pd.read_csv(unfall_table_path)
few_shot_llm_labels = pd.read_csv(few_shot_llm_labels_path)
finetuned_xlmr_trained_w_low_quality_predictions = pd.read_csv(finetuned_xlmr_trained_w_low_quality_predictions_path)
finetuned_xlmr_trained_w_high_quality_predictions = pd.read_csv(finetuned_xlmr_trained_w_high_quality_predictions_path)
multimodal_model_trained_w_high_quality_predictions = pd.read_csv(multimodal_model_trained_w_high_quality_predictions_path)
multimodal_model_trained_w_low_quality_predictions = pd.read_csv(multimodal_model_trained_w_low_quality_predictions_path)
tabular_model_trained_w_high_quality_predictions = pd.read_csv(tabular_model_trained_w_high_quality_predictions_path)
tabular_model_trained_w_low_quality_predictions = pd.read_csv(tabular_model_trained_w_low_quality_predictions_path)
translated_unfalltext_table = pd.read_csv(translated_unfalltext_table_path)

# Remove duplicated UN_KEY rows
test_set = test_set.drop_duplicates(subset="UN_KEY")
# Get only new test set with 120 observations
# test_set = test_set[test_set["new_test_set_expert_label"].notnull()]
test_set = test_set[test_set["old_test_set_expert_label"].notnull()]

print("Test Set Size: ", test_set.shape)

In [ ]:
# Update column names
tabular_model_trained_w_high_quality_predictions.rename(columns={"tabular_model_prediction": "tabular_model_trained_w_high_quality_prediction"}, inplace=True)
tabular_model_trained_w_low_quality_predictions.rename(columns={"tabular_model_prediction": "tabular_model_trained_w_low_quality_prediction"}, inplace=True)
multimodal_model_trained_w_high_quality_predictions.rename(columns={"multimodal_model_prediction": "multimodal_model_trained_w_high_quality_prediction"}, inplace=True)
multimodal_model_trained_w_low_quality_predictions.rename(columns={"multimodal_model_prediction": "multimodal_model_trained_w_low_quality_prediction"}, inplace=True)
finetuned_xlmr_trained_w_low_quality_predictions.rename(columns={"finetuned_xlmr_predictions": "finetuned_xlmr_trained_w_low_quality_prediction"}, inplace=True)
finetuned_xlmr_trained_w_high_quality_predictions.rename(columns={"finetuned_xlmr_predictions": "finetuned_xlmr_trained_w_high_quality_prediction"}, inplace=True)

In [ ]:
# Merge all the dataframes
merged_df = test_set.merge(few_shot_llm_labels, on="UN_KEY", how="left").merge(
    tabular_model_trained_w_high_quality_predictions, on="UN_KEY", how="left").merge(
    tabular_model_trained_w_low_quality_predictions, on="UN_KEY", how="left").merge(
    translated_unfalltext_table[["UN_KEY", "Translated_Text"]], on="UN_KEY", how="left").merge(
    unfall_table[["UN_KEY", "Unfalltyp"]], on="UN_KEY", how="left").merge(
    multimodal_model_trained_w_high_quality_predictions, on="UN_KEY", how="left").merge(
    multimodal_model_trained_w_low_quality_predictions, on="UN_KEY", how="left").merge(
    finetuned_xlmr_trained_w_low_quality_predictions, on="UN_KEY", how="left").merge(
    finetuned_xlmr_trained_w_high_quality_predictions, on="UN_KEY", how="left")

merged_df.rename(columns={"Unfalltyp": "human_label"}, inplace=True)

merged_df.shape

In [ ]:
merged_df.head(3)

In [ ]:
merged_df.info()

In [ ]:
# True Label Distr
merged_df.expert_label.value_counts()

In [ ]:
# Count occurrences of each unique label
label_counts = merged_df["expert_label"].value_counts().sort_index()

# Convert integer labels to string descriptions using custom_palette keys
label_names = list(custom_palette.keys())

# Set figure size
plt.figure(figsize=(10, 6))

# Plot bar chart with custom colors
sns.barplot(x=label_names, y=label_counts.values, palette=custom_palette)

# Labels and title
plt.xlabel("Expert Labels", fontsize=14)
plt.ylabel("Count")
plt.title("Accident Type Distribution in Test Dataset", fontsize=16)
plt.xticks(rotation=45, ha="right")  # Rotate x labels for better readability

# Show plot
plt.show()


In [ ]:
# Extract the relevant columns
true_labels = merged_df['expert_label']
human_labels = merged_df['human_label']
llm_few_shot_labels = merged_df['few_shot_LLM_Labels']
finetuned_xlmr_trained_w_low_quality_labels = merged_df['finetuned_xlmr_trained_w_low_quality_prediction']
finetuned_xlmr_trained_w_high_quality_labels = merged_df['finetuned_xlmr_trained_w_high_quality_prediction']
multimodal_model_trained_w_high_quality_labels = merged_df['multimodal_model_trained_w_high_quality_prediction']
multimodal_model_trained_w_low_quality_labels = merged_df['multimodal_model_trained_w_low_quality_prediction']
tabular_model_trained_w_high_quality_labels = merged_df['tabular_model_trained_w_high_quality_prediction']
tabular_model_trained_w_low_quality_labels = merged_df['tabular_model_trained_w_low_quality_prediction']
# old_project_labels = merged_df['old_project_label']

# Calculate accuracy
accuracy_llm = accuracy_score(true_labels, llm_few_shot_labels)
accuracy_human = accuracy_score(true_labels, human_labels)
accuracy_finetuned_xlmr_trained_w_low_quality = accuracy_score(true_labels, finetuned_xlmr_trained_w_low_quality_labels)
accuracy_finetuned_xlmr_trained_w_high_quality = accuracy_score(true_labels, finetuned_xlmr_trained_w_high_quality_labels)
accuracy_multimodal_model_trained_w_high_quality = accuracy_score(true_labels, multimodal_model_trained_w_high_quality_labels)
accuracy_multimodal_model_trained_w_low_quality = accuracy_score(true_labels, multimodal_model_trained_w_low_quality_labels)
accuracy_tabular_model_trained_w_high_quality = accuracy_score(true_labels, tabular_model_trained_w_high_quality_labels)
accuracy_tabular_model_trained_w_low_quality = accuracy_score(true_labels, tabular_model_trained_w_low_quality_labels)
# accuracy_old_project = accuracy_score(true_labels, old_project_labels)

# Calculate F1 scores (macro-averaged for multiclass)
f1_llm = f1_score(true_labels, llm_few_shot_labels, average='weighted')
f1_human = f1_score(true_labels, human_labels, average='weighted')
f1_finetuned_xlmr_trained_w_low_quality = f1_score(true_labels, finetuned_xlmr_trained_w_low_quality_labels, average='weighted')
f1_finetuned_xlmr_trained_w_high_quality = f1_score(true_labels, finetuned_xlmr_trained_w_high_quality_labels, average='weighted')
f1_multimodel_model_trained_w_high_quality = f1_score(true_labels, multimodal_model_trained_w_high_quality_labels, average='weighted')
f1_multimodel_model_trained_w_low_quality = f1_score(true_labels, multimodal_model_trained_w_low_quality_labels, average='weighted')
f1_tabular_model_trained_w_high_quality = f1_score(true_labels, tabular_model_trained_w_high_quality_labels, average='weighted')
f1_tabular_model_trained_w_low_quality = f1_score(true_labels, tabular_model_trained_w_low_quality_labels, average='weighted')
# f1_old_project = f1_score(true_labels, old_project_labels, average='weighted')

# Print the results
print(f"Accuracy (LLM Few-Shot): {accuracy_llm:.4f}")
print(f"Accuracy (Human Label): {accuracy_human:.4f}")
print(f"Accuracy (Finetuned XLM-R Trained w Low Quality): {accuracy_finetuned_xlmr_trained_w_low_quality:.4f}")
print(f"Accuracy (Finetuned XLM-R Trained w High Quality): {accuracy_finetuned_xlmr_trained_w_high_quality:.4f}")
print(f"Accuracy (Multimodal Model Trained w High Quality): {accuracy_multimodal_model_trained_w_high_quality:.4f}")
print(f"Accuracy (Multimodal Model Trained w Low Quality): {accuracy_multimodal_model_trained_w_low_quality:.4f}")
print(f"Accuracy (Tabular Model Trained w High Quality): {accuracy_tabular_model_trained_w_high_quality:.4f}")
print(f"Accuracy (Tabular Model Trained w Low Quality): {accuracy_tabular_model_trained_w_low_quality:.4f}")
# print(f"Accuracy (Old Project): {accuracy_old_project:.4f}")
print(f"Weighted F1 Score (LLM Few-Shot): {f1_llm:.4f}")
print("Weighted F1 Score (Human Label):", f1_human)
print(f"Weighted F1 Score (Finetuned XLM-R Trained w Low Quality): {f1_finetuned_xlmr_trained_w_low_quality:.4f}")
print(f"Weighted F1 Score (Finetuned XLM-R Trained w High Quality): {f1_finetuned_xlmr_trained_w_high_quality:.4f}")
print(f"Weighted F1 Score (Multimodal Model Trained w High Quality): {f1_multimodel_model_trained_w_high_quality:.4f}")
print(f"Weighted F1 Score (Multimodal Model Trained w Low Quality): {f1_multimodel_model_trained_w_low_quality:.4f}")
print(f"Weighted F1 Score (Tabular Model Trained w High Quality): {f1_tabular_model_trained_w_high_quality:.4f}")
print(f"Weighted F1 Score (Tabular Model Trained w Low Quality): {f1_tabular_model_trained_w_low_quality:.4f}")
# print(f"Weighted F1 Score (Old Project): {f1_old_project:.4f}")

In [ ]:
# Ensure the labels are sorted from 1 to 7
sorted_labels = sorted(true_labels.unique())

# Plot LLM Few-Shot Confusion Matrix
cm_human = confusion_matrix(true_labels, human_labels, labels=sorted_labels)
disp_human = ConfusionMatrixDisplay(confusion_matrix=cm_human, display_labels=sorted_labels)
disp_human.plot(cmap='Blues', xticks_rotation='vertical')
plt.title("Confusion Matrix - Human Labels")
plt.show()

In [ ]:
# Compute Precision Matrix (Normalize each column)
recall_matrix_human = cm_human / np.sum(cm_human, axis=1, keepdims=True)

# Handle division by zero (if any class was never predicted)
recall_matrix_human = np.nan_to_num(recall_matrix_human)

# Plot Precision Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(recall_matrix_human, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=sorted_labels, yticklabels=sorted_labels)

plt.xlabel("Human Labels")
plt.ylabel("Expert Labels")
plt.title("Recall Matrix - Human Labels")
plt.show()

In [ ]:
# Ensure the labels are sorted from 1 to 7
sorted_labels = sorted(true_labels.unique())

# Plot LLM Few-Shot Confusion Matrix
cm_llm = confusion_matrix(true_labels, llm_few_shot_labels, labels=sorted_labels)
disp_llm = ConfusionMatrixDisplay(confusion_matrix=cm_llm, display_labels=sorted_labels)
disp_llm.plot(cmap='Blues', xticks_rotation='vertical')
plt.title("Confusion Matrix - LLM Few-Shot")
plt.show()

In [ ]:
# Plot XLMR Finetuned Confusion Matrix
cm_xlmr_finetuned = confusion_matrix(true_labels, finetuned_llm_labels, labels=sorted_labels)
disp_llm = ConfusionMatrixDisplay(confusion_matrix=cm_xlmr_finetuned, display_labels=sorted_labels)
disp_llm.plot(cmap='Blues', xticks_rotation='vertical')
plt.title("Confusion Matrix - Finetuned LLM")
plt.show()

In [ ]:
# Plot Multimodal Model Confusion Matrix
cm_multimodal = confusion_matrix(true_labels, multimodal_model_trained_w_high_quality_labels, labels=sorted_labels)
disp_llm = ConfusionMatrixDisplay(confusion_matrix=cm_multimodal, display_labels=sorted_labels)
disp_llm.plot(cmap='Blues', xticks_rotation='vertical')
plt.title("Confusion Matrix - Multimodal Model")
plt.show()

In [ ]:
# Compute Precision Matrix (Normalize each column)
recall_matrix_multimodal = cm_multimodal / np.sum(cm_multimodal, axis=1, keepdims=True)

# Handle division by zero (if any class was never predicted)
recall_matrix_multimodal = np.nan_to_num(recall_matrix_multimodal)

# Plot Precision Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(recall_matrix_multimodal, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=sorted_labels, yticklabels=sorted_labels)

plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Recall Matrix - Multimodal Model")
plt.show()

In [ ]:
# Plot Tabular Model Confusion Matrix
cm_tabular = confusion_matrix(true_labels, tabular_model_labels, labels=sorted_labels)
disp_llm = ConfusionMatrixDisplay(confusion_matrix=cm_tabular, display_labels=sorted_labels)
disp_llm.plot(cmap='Blues', xticks_rotation='vertical')
plt.title("Confusion Matrix - Tabular Model")
plt.show()

In [ ]:
# Compute Precision Matrix (Normalize each column)
recall_matrix_tabular = cm_tabular / np.sum(cm_tabular, axis=1, keepdims=True)

# Handle division by zero (if any class was never predicted)
recall_matrix_tabular = np.nan_to_num(recall_matrix_tabular)

# Plot Precision Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(recall_matrix_tabular, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=sorted_labels, yticklabels=sorted_labels)

plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Recall Matrix - Tabular Model")
plt.show()

In [ ]:
# Plot Old Project Confusion Matrix
cm_old = confusion_matrix(true_labels, old_project_labels, labels=sorted_labels)
disp_old_project = ConfusionMatrixDisplay(confusion_matrix=cm_old, display_labels=sorted_labels)
disp_old_project.plot(cmap='Blues', xticks_rotation='vertical')
plt.title("Confusion Matrix - Old Project")
plt.show()

In [ ]:
merged_df[merged_df.true_label != merged_df.multimodal_model_prediction][['UN_KEY', 'true_label', 'Translated_Text', 'multimodal_model_prediction']]

# Human Label 7 Analysis

In [ ]:
# Data Paths
unfall_table_path = "data/D_Unfall_RData.csv"
unfalltext_table_path = "data/D_Unfalltext_image.csv"
translated_unfalltext_table_path = "data/translated_unfalltext_0_103111.csv"
few_shot_llm_predicted_labels_table_path = "data/few_shot_llm_predicted_labels_unfalltext_0_103111.csv"
multimodal_model_all_data_predictions_path = "data/multimodal_model_predictions_all_data.csv"

In [ ]:
unfall_table = pd.read_csv(unfall_table_path)
unfalltext_table = pd.read_csv(unfalltext_table_path)
translated_unfalltext_table = pd.read_csv(translated_unfalltext_table_path)
few_shot_llm_predicted_labels_table = pd.read_csv(few_shot_llm_predicted_labels_table_path)
multimodal_model_all_data_predictions = pd.read_csv(multimodal_model_all_data_predictions_path)

In [ ]:
# Merge the two tables on UN_KEY
unfall_text_predictions_merged = unfall_table.merge(
    unfalltext_table, on="UN_KEY", how="left").merge(
        few_shot_llm_predicted_labels_table[["UN_KEY", "few_shot_LLM_Labels"]], on="UN_KEY", how="left").merge(
            multimodal_model_all_data_predictions[["UN_KEY", "multimodal_model_prediction"]], on="UN_KEY", how="left").merge(
                translated_unfalltext_table[["UN_KEY", "Translated_Text"]], on="UN_KEY", how="left")

# Filter only the Human Label = 7
unfall_text_predictions_merged_human_label_7 = unfall_text_predictions_merged[unfall_text_predictions_merged.Unfalltyp==7].copy().reset_index(drop=True)
# Drop not predicted rows by multimodal model
unfall_text_predictions_merged_human_label_7 = unfall_text_predictions_merged_human_label_7[unfall_text_predictions_merged_human_label_7.multimodal_model_prediction.notnull() & unfall_text_predictions_merged_human_label_7.multimodal_model_prediction.notnull()]
unfall_text_predictions_merged_human_label_7.shape

In [ ]:
unfall_text_predictions_merged_human_label_7.head(3)

In [ ]:
unfall_text_predictions_merged_human_label_7.multimodal_model_prediction.value_counts(dropna=False)

In [ ]:
unfall_text_predictions_merged_human_label_7.few_shot_LLM_Labels.value_counts()

In [ ]:
# Bubble plot for Human Label vs Multimodal Model Prediction
multimodal_labels_per_human_label = unfall_text_predictions_merged_human_label_7.groupby('Unfalltyp')['multimodal_model_prediction'].value_counts().unstack().fillna(0)
multimodal_labels_per_human_label_percentage = (multimodal_labels_per_human_label.div(multimodal_labels_per_human_label.sum(axis=1), axis=0) * 100).round(0)

# Prepare the data (keeping only integer values for axes)
distr_plot_data = multimodal_labels_per_human_label_percentage.copy()
categories = [int(x) for x in distr_plot_data.index]  # Only integers
llm_labels = [int(x) for x in distr_plot_data.columns]  # Only integers

# Set up the plot
fig, ax = plt.subplots(figsize=(6, 12))
ax.set_xlim(-1, len(categories))
ax.set_ylim(-1, len(llm_labels))

# Create the bubble chart
for i, category in enumerate(categories):
    for j, label in enumerate(llm_labels):
        percentage = distr_plot_data.iloc[i, j]
        if percentage > 0:  # Only plot non-zero percentages
            circle_size = percentage * 100  # Adjust size scaling factor
            ax.scatter(i, j, s=circle_size, color='skyblue', alpha=0.7, edgecolor='black')
            circle_font = 'bold' if percentage >= 20 else 'normal'
            circle_font_size = 20 if percentage >= 20 else 10
            ax.text(i, j, f"{int(percentage)}%", ha='center', va='center', 
                    fontsize=circle_font_size, weight=circle_font)

# Adjust ticks and labels (only integer values)
ax.set_xticks(range(len(categories)))
ax.set_xticklabels(categories, fontsize=20)
ax.set_xlabel("Human Labels", fontsize=20)

ax.set_yticks(range(len(llm_labels)))
ax.set_yticklabels(llm_labels, fontsize=20)
ax.set_ylabel("Multimodal Model Labels", fontsize=20)

# # Add a separate legend for category descriptions
# legend_patches = [mpatches.Patch(color='white', label=f"{int(k)}: {v}") 
#                   for k, v in unfalltyp_mapping_eng.items()]
# # ax.legend(handles=legend_patches, loc='upper left', fontsize=16, title="Label Mappings")

# Add grid for better readability
ax.grid(True, linestyle='--', alpha=0.6)

# Add title
plt.title("Multimodal Model Labels vs Human Labels", fontsize=20)
plt.tight_layout()
plt.show()


In [ ]:
# Count occurrences of each unique label
label_counts = unfall_text_predictions_merged_human_label_7["multimodal_model_prediction"].value_counts().sort_index()

# Convert counts to percentages
total_count = label_counts.sum()
label_percentages = (label_counts / total_count) * 100

# Convert integer labels to string descriptions using custom_palette keys
label_names = list(custom_palette.keys())

# Set figure size
plt.figure(figsize=(12, 6))

# Plot bar chart with custom colors
ax = sns.barplot(x=label_names, y=label_percentages.values, palette=custom_palette)

# Labels and title
plt.xlabel("Accident Type")
plt.ylabel("Percentage (%)")
plt.title("Accident Type Distribution in Multimodal Model Predictions for Human Label 7 Accidents", fontsize=16)
plt.xticks(rotation=45, ha="right")  # Rotate x labels for better readability

# Annotate bars with percentage values
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", 
                (p.get_x() + p.get_width() / 2, p.get_height()), 
                ha='center', va='bottom', fontsize=12)

# Show plot
plt.show()

# Human Label 1-6 Analysis

In [ ]:
unfall_text_predictions_merged_human_label_1_6_llm_few_shot_different = unfall_text_predictions_merged[(unfall_text_predictions_merged.Unfalltyp.isin([1, 2, 3, 4, 5, 6])) & (unfall_text_predictions_merged.few_shot_LLM_Labels != unfall_text_predictions_merged.Unfalltyp)].copy().reset_index(drop=True)
unfall_text_predictions_merged_human_label_1_6_llm_few_shot_different.shape

In [ ]:
# unfall_text_predictions_merged_human_label_1_6_llm_few_shot_different.to_csv("data/multimodal_predictions_for_human_label_1_6_subset.csv", index=False)

In [ ]:
unfall_text_predictions_merged_human_label_1_6_llm_few_shot_different.head(3)

In [ ]:
unfall_text_predictions_merged_human_label_1_6_llm_few_shot_different.shape

In [ ]:
unfall_text_predictions_merged_human_label_1_6_llm_few_shot_different[unfall_text_predictions_merged_human_label_1_6_llm_few_shot_different.Unfalltyp==\
                                                                      unfall_text_predictions_merged_human_label_1_6_llm_few_shot_different.multimodal_model_prediction].shape

In [ ]:
# Multimodal Model Prediction == Human Label
unfall_text_predictions_merged_human_label_1_6_llm_few_shot_different[unfall_text_predictions_merged_human_label_1_6_llm_few_shot_different.Unfalltyp==\
                                                                      unfall_text_predictions_merged_human_label_1_6_llm_few_shot_different.multimodal_model_prediction].shape[0] / \
                                                                        unfall_text_predictions_merged_human_label_1_6_llm_few_shot_different.shape[0]

In [ ]:
# Count occurrences of each unique label
label_counts = unfall_text_predictions_merged_human_label_1_6_llm_few_shot_different["multimodal_model_prediction"].value_counts().sort_index()

# Convert counts to percentages
total_count = label_counts.sum()
label_percentages = (label_counts / total_count) * 100

# Convert integer labels to string descriptions using custom_palette keys
label_names = list(custom_palette.keys())

# Set figure size
plt.figure(figsize=(12, 6))

# Plot bar chart with custom colors
ax = sns.barplot(x=label_names, y=label_percentages.values, palette=custom_palette)

# Labels and title
plt.xlabel("Accident Type")
plt.ylabel("Percentage (%)")
plt.title("Accident Type Distribution in Multimodal Model Predictions for Human Label 1-6 Accidents not matching LLM Few Shot Labels", fontsize=16)
plt.xticks(rotation=45, ha="right")  # Rotate x labels for better readability

# Annotate bars with percentage values
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", 
                (p.get_x() + p.get_width() / 2, p.get_height()), 
                ha='center', va='bottom', fontsize=12)

# Show plot
plt.show()